In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import os

# --- 1. CONFIGURAZIONE SPLIT ---
# Indici forniti per training e validation
train_indices = [22, 0, 1, 2, 3, 10, 14, 16, 17, 18, 19, 21, 8, 9, 12, 5, 6, 4]
val_indices = [23, 7, 11, 13, 15, 20]

def load_data_from_windows(indices, base_path="dataset/data/"):
    x_data = []
    y_data = []
    
    for idx in indices:
        file_path = os.path.join(base_path, f"window_{idx:06d}.npz")
        if os.path.exists(file_path):
            data = np.load(file_path)
            # Carichiamo i dati grezzi e le label (coordinate x,y)
            x_data.append(data['radar_cir_iq'])  # Shape: (T, 6, 3, 120, 2)
            y_data.append(data['people_coords']) # Shape: (T, 4, 2) - coordinate x,y per 4 persone
            
    return x_data, y_data

# --- 2. PRE-PROCESSING (Magnitude + EMA Decluttering) ---
def preprocess_batch(x_list, alpha=0.05):
    processed_x = []
    for window in x_list:
        # Calcolo Magnitudo: sqrt(I^2 + Q^2) -> Shape: (T, 6, 3, 120)
        mag = np.sqrt(window[..., 0]**2 + window[..., 1]**2)
        
        # EMA Decluttering (rimozione riflessi statici come da slide)
        bg = np.zeros_like(mag[0])
        decluttered = []
        for frame in mag:
            bg = alpha * frame + (1 - alpha) * bg
            decluttered.append(frame - bg)
            
        # Reshape per la CNN: uniamo radar e antenne come canali o feature
        # Qui trattiamo ogni frame come (120 range_bins, 18 feature) 
        # (6 radar * 3 antenne = 18 canali di informazione)
        frame_data = np.array(decluttered).transpose(0, 3, 1, 2).reshape(-1, 120, 18)
        processed_x.append(frame_data)
        
    return np.vstack(processed_x)

# --- 3. DEFINIZIONE DELLA RETE (Stile Lecture 5 CNN) ---
def build_model(input_shape):
    model = models.Sequential([
        # Input Layer: (Range Bins, Features/Channels)
        layers.Input(shape=input_shape),
        
        # Primo Blocco Convoluzionale (Conv2D richiede 3D, usiamo Conv1D per segnali 1D)
        # O usiamo Conv2D aggiungendo una dimensione fittizia per simulare un'immagine
        layers.Reshape((input_shape[0], input_shape[1], 1)),
        
        layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        
        # Global Average Pooling per risparmiare SRAM (visto nei Lab per Edge AI)
        layers.GlobalAveragePooling2D(),
        
        # Parte Dense per la regressione delle coordinate
        layers.Dense(64, activation='relu'),
        
        # Output: 8 valori (x1, y1, x2, y2, x3, y3, x4, y4)
        layers.Dense(8, activation='linear') 
    ])
    
    # Loss: Mean Absolute Error (MAE) o MSE, tipico per la regressione nei Lab
    model.compile(optimizer='adam', loss='mae', metrics=['mse'])
    return model

# --- ESECUZIONE ---
# 1. Caricamento
x_train_raw, y_train_raw = load_data_from_windows(train_indices)
x_val_raw, y_val_raw = load_data_from_windows(val_indices)

# 2. Pre-processing
x_train = preprocess_batch(x_train_raw)
y_train = np.vstack(y_train_raw).reshape(-1, 8) # Flatten coordinate

x_val = preprocess_batch(x_val_raw)
y_val = np.vstack(y_val_raw).reshape(-1, 8)

# 3. Creazione Modello
# Ogni input è (120 range bins, 18 feature)
input_dim = (120, 18)
model = build_model(input_dim)

model.summary()

# --- 4. VALUTAZIONE VINCOLI (Stile Lab 4/5) ---
# Calcolo parametri e stima memoria Flash
total_params = model.count_params()
estimated_flash_kb = (total_params * 4) / 1024  # Assumendo float32 (4 bytes)
print(f"\nParametri Totali: {total_params}")
print(f"Stima Flash (float32): {estimated_flash_kb:.2f} KB")